**Caqso de estudio**

Una fintech quiere estimar el error de sus datos internos asociados a las operaciones portuarias y logísticas de contenedores en diferentes puertos del mundo. Adicionalmente, busca evaluar, mediante la teoría de credibilidad, qué nivel de confianza puede asignarse a una base de datos externa de operaciones perteneciente a otra empresa naviera, con el fin de determinar si dicha información puede integrarse de manera confiable a su base de datos observada.

0. se procede con la carga de librerias de trabajo

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive

1. se procede con la estimacion del error de los datos internos


In [ ]:
z=2
P=0.95

In [ ]:
path ="/content/drive/MyDrive/Integracion de datos/1. RiesgoOperacional_EVERGREEEN.xlsx"



XDB=pd.read_excel(path)

display (XDB)

Freq=XDB.iloc[:,3]; Sev=XDB.iloc[:,4]
LDA=Freq*Sev
ND=len(LDA)

#Se estima el intervalo de confianza
u=np.mean(LDA); sigma=np.std(LDA)
print("El limite inferior de los datos es:",u-2*sigma)
print("El limite superiror de los datos es",u+2*sigma)

# el error de mis datos a una confiabilidad del 95% es (usando P mayúscula)
E=z*np.sqrt((P*(1-P))/ND)
print("el error en el calculo de mis datos es:",E)

#4. Que pasa con el error y la media:
print("El intevalo de la media con el error es:")
print("La media menos el error",(u-E)*1000)
print("La media más el error",(u+E)*1000)
print("El desfase en dolares es:",(u+E)*1000-(u-E)*1000)

**Análisis de Resultados**
El error encontrado en los datos se acerca a los 16.46 dolareas por evento de riesgo. Suponiendo un total de 1,000 eventos de riesgo, o 1,000 contenedores que se quedaron en tierra, se tiene que tener una provisión adicional en el presupuesto de las pérdidas de 1,000 Contenedor*16.46 USD/Contenedor. De manera general, la media de la pérdida por contenedor se ubica en el intevalo [231258.37067397078,231274.8340080642]. En este sentido el porcentaje de error esta por debajo del 1%, lo que es bueno de acuerdo con el número de datos disponible. Es importante que el valor de p se debe estimar en función de la calidad de los datos (100 solicitudes de crédito - 95 son correctas - 5 hay que llamar al cliente para que traiga más documentos o mejorar la información)

In [ ]:
#Se procede con la carga de los datos observados
path2= "/content/drive/MyDrive/Integracion de datos/3. Datos COSCO Shipping.xlsx"
XDB=pd.read_excel(path2)
#display(XDB)

#1. Los datos que caracterizan el riesgo
Freq=XDB.iloc[:,3]; Sev=XDB.iloc[:,4]
LDA=Freq*Sev; ND=len(LDA); sigmao=np.std(LDA); uo=np.mean(LDA)

#2. Cargamos los datos externos (corregido para usar la ruta de Drive)
nxle= path2
XDE=pd.read_excel(nxle)
Freqe=XDE.iloc[:,3]; Seve=XDE.iloc[:,4]
LDAe=Freqe*Seve; NDe=len(LDAe);sigmae=np.std(LDAe);ue=np.mean(LDAe)

#Varianza Ponderada
EPV=(ND*sigmao**2+NDe*sigmae**2)/(ND+NDe)
print("La ponderación de las varianzas es:",EPV)

#Media Hipotetica - Ponderado de las medias
uh=(ND*uo+NDe*ue)/(ND+NDe)
print("La media ponderada de los datos es:",uh)

#La varianza de dos variables es:
VHM=(((ND)/(ND+NDe))*uo**2+((NDe)/(ND+NDe))*ue**2)-uh**2
print("La varianza de dos variables es:",VHM)
Cr=ND/(ND+(EPV/VHM))
print("La credibilidad de mis datos es:",Cr*100)
print("La credibilidad de la base de datos externos es del:",(1-Cr)*100)

**Análisis de Resultados**
La forma de interpretar la decisión del modelo está en la fórmula final:
𝐶𝑟=(𝑁𝐷_𝑜)/(𝑁𝐷_𝑜+𝐸𝑃𝑉/𝑉𝐻𝑀)
Esta fracción es una balanza entre el volumen de tu información y la relación ruido/señal:
Si el ruido supera a la señal (𝐸𝑃𝑉>𝑉𝐻𝑀): El término 𝐸𝑃𝑉/𝑉𝐻𝑀se vuelve muy grande. Esto infla el denominador y empuja tu credibilidad 𝐶𝑟hacia 0. El modelo te está diciendo que tus datos internos son demasiado volátiles o erráticos para ser confiables por sí solos, por lo que te obligará a darle más peso a la base de datos externa (𝐿𝐷𝐴_𝑒) para calcular tu riesgo.
Si la señal es clara y fuerte (𝑉𝐻𝑀>𝐸𝑃𝑉): El término 𝐸𝑃𝑉/𝑉𝐻𝑀se vuelve pequeño. El denominador se acerca al valor del numerador (𝑁𝐷_𝑜), empujando 𝐶𝑟hacia 1. El modelo determina que tus datos internos son estables y consistentes, por lo que puedes confiar plenamente en ellos sin diluirlos con datos externos.
El peso de la muestra (𝑁𝐷_𝑜): A medida que el volumen de tus observaciones internas (𝑁𝐷_𝑜) crece hacia el infinito, este número abruma al factor 𝐸𝑃𝑉/𝑉𝐻𝑀en el denominador. Matemáticamente, 𝐶𝑟tenderá a 1. Es el principio de la ley de los grandes números: si tienes una cantidad masiva de datos históricos, el modelo confiará en tu propia distribución 𝐿𝐷𝐴, asumiendo que ya contiene suficiente información para absorber cualquier volatilidad.
